# SCRIBE Local Pipeline Demo

Step-by-step walkthrough of the OCR pipeline on two images — one easy, one hard —
showing how the auto-detection router adapts preprocessing to each image's needs.
Followed by a summary comparison across all five test images.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import yaml
from matplotlib import pyplot as plt

from scribe.preprocessing import analyze_image, preprocess
from scribe.ocr import run_tesseract
from scribe.postprocessing import postprocess

project_root = Path("..").resolve()
config = yaml.safe_load(open(project_root / "configs" / "pipeline.yaml"))
manifest = yaml.safe_load(open(project_root / "data" / "images.yaml"))["images"]
images_dir = project_root / "data" / "images"
accept_thresh = config["confidence_thresholds"]["accept"]
review_thresh = config["confidence_thresholds"]["review"]

def show(title, img, figsize=(12, 16)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

def show_compare(title_a, img_a, title_b, img_b, figsize=(18, 12)):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    for ax, title, img in [(ax1, title_a, img_a), (ax2, title_b, img_b)]:
        ax.imshow(img, cmap="gray")
        ax.set_title(title, fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def run_single_image(name):
    """Run the full pipeline on one image. Returns all intermediate results."""
    meta = manifest[name]
    img_bgr = cv2.imread(str(images_dir / meta["filename"]))
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    recipe = analyze_image(gray, config)
    preprocessed = preprocess(gray, recipe)

    raw_r = run_tesseract(gray)
    smart_r = run_tesseract(preprocessed)

    best = "smart" if smart_r["avg_confidence"] > raw_r["avg_confidence"] else "raw"
    best_r = smart_r if best == "smart" else raw_r
    conf = best_r["avg_confidence"]

    if conf >= accept_thresh:
        routing = "ACCEPT"
    elif conf >= review_thresh:
        routing = "REVIEW"
    else:
        routing = "ESCALATE"

    post = postprocess(best_r["text"])

    return {
        "name": name, "meta": meta, "gray": gray, "preprocessed": preprocessed,
        "recipe": recipe, "raw_r": raw_r, "smart_r": smart_r,
        "best": best, "best_r": best_r, "routing": routing, "post": post,
    }

## Image 1: `rattler_page` (easy — clean, high-res)

A book page at 2417×3467 px. Clean scan, high contrast. The router should
decide: no upscale needed, skip CLAHE, no deskew. Raw OCR should already be good.

In [ ]:
r1 = run_single_image("rattler_page")
gray1, recipe1 = r1["gray"], r1["recipe"]
h, w = gray1.shape
print(f"Image: {r1['meta']['filename']}  ({w}×{h} px)")
print(f"Notes: {r1['meta']['notes']}")
show(f"rattler_page ({w}×{h})", gray1, figsize=(8, 11))

### Auto-detection and results

In [ ]:
print(f"Router decisions:")
print(f"  Scale:  {recipe1['signals']['short_side']}px short side → {recipe1['scale']}x (target {recipe1['signals']['target_short_side']}px)")
print(f"  CLAHE:  contrast std={recipe1['signals']['contrast_std']} → apply={recipe1['apply_clahe']}")
print(f"  Deskew: angle={recipe1['skew_angle']}° → apply={abs(recipe1['skew_angle']) > 0.3}")
print(f"\nOCR results:")
print(f"  Raw:   {r1['raw_r']['avg_confidence']:.1f}% ({r1['raw_r']['word_count']} words)")
print(f"  Smart: {r1['smart_r']['avg_confidence']:.1f}% ({r1['smart_r']['word_count']} words)")
print(f"  Best:  {r1['best']}  →  Routing: {r1['routing']}")
if r1["post"]["fields"]:
    print(f"\nExtracted fields: {json.dumps(r1['post']['fields'], indent=2)}")
print(f"\nFirst 300 chars of extracted text:")
print(r1["best_r"]["text"][:300])

In [ ]:
from scribe.visualization import render_side_by_side

font_size1 = max(10, r1["gray"].shape[0] // 120)
# Use the image that was actually OCR'd (raw won for rattler)
ocr_img1 = r1["gray"] if r1["best"] == "raw" else r1["preprocessed"]
sbs1 = render_side_by_side(ocr_img1, r1["best_r"]["words"], font_size=font_size1)

fig, ax = plt.subplots(figsize=(20, 14))
ax.imshow(sbs1)
ax.set_title("Original (left) vs OCR extraction colored by confidence (right)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## Image 2: `grocery_contract` (hard — low-res, faded, 1903)

A 1903 legal contract at 620×628 px. Yellowed paper, faded ink, low resolution.
The router should apply all preprocessing: upscale, CLAHE, and check for deskew.
This is the kind of document that separates a good OCR pipeline from a naive one.

In [ ]:
r2 = run_single_image("grocery_contract")
gray2, recipe2, prep2 = r2["gray"], r2["recipe"], r2["preprocessed"]
h, w = gray2.shape
print(f"Image: {r2['meta']['filename']}  ({w}×{h} px)")
print(f"Notes: {r2['meta']['notes']}")
show_compare(f"Raw ({w}×{h})", gray2,
             f"Preprocessed ({prep2.shape[1]}×{prep2.shape[0]})", prep2)

### Auto-detection, OCR, and extraction

In [ ]:
print(f"Router decisions:")
print(f"  Scale:  {recipe2['signals']['short_side']}px short side → {recipe2['scale']}x")
print(f"  CLAHE:  contrast std={recipe2['signals']['contrast_std']} → apply={recipe2['apply_clahe']}")
print(f"  Deskew: angle={recipe2['skew_angle']}° → apply={abs(recipe2['skew_angle']) > 0.3}")
print(f"\nOCR results:")
print(f"  Raw:   {r2['raw_r']['avg_confidence']:.1f}% ({r2['raw_r']['word_count']} words)")
print(f"  Smart: {r2['smart_r']['avg_confidence']:.1f}% ({r2['smart_r']['word_count']} words)")
print(f"  Δ:     +{r2['smart_r']['avg_confidence'] - r2['raw_r']['avg_confidence']:.1f}%")
print(f"  Best:  {r2['best']}  →  Routing: {r2['routing']}")
print(f"\nExtracted fields:")
print(json.dumps(r2["post"]["fields"], indent=2))
print(f"\nFirst 400 chars of extracted text:")
print(r2["best_r"]["text"][:400])

### Word-level confidence visualization (grocery_contract)

In [ ]:
vis = cv2.cvtColor(prep2, cv2.COLOR_GRAY2RGB)
for word in r2["best_r"]["words"]:
    x, y, bw, bh = word["bbox"]["x"], word["bbox"]["y"], word["bbox"]["w"], word["bbox"]["h"]
    c = word["conf"]
    color = (0, 200, 0) if c >= 80 else (0, 200, 200) if c >= 50 else (0, 0, 200)
    cv2.rectangle(vis, (x, y), (x + bw, y + bh), color, 2)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(vis)
ax.set_title("Word confidence: green ≥80%, yellow ≥50%, red <50%", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

confs = [w["conf"] for w in r2["best_r"]["words"]]
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(confs, bins=20, range=(0, 100), color="steelblue", edgecolor="white")
ax.axvline(review_thresh, color="orange", ls="--", label=f"Review ({review_thresh}%)")
ax.axvline(accept_thresh, color="green", ls="--", label=f"Accept ({accept_thresh}%)")
ax.set_xlabel("Word confidence (%)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

### OCR text overlay — original vs extracted

The left side shows the preprocessed image. The right side renders each extracted
word at its detected position, colored by confidence (green = high, red = low).
This makes it easy to see where OCR succeeded and where it struggled.

In [ ]:
from scribe.visualization import render_side_by_side

# Scale font relative to image size
font_size = max(10, prep2.shape[0] // 120)

side_by_side = render_side_by_side(prep2, r2["best_r"]["words"], font_size=font_size)

fig, ax = plt.subplots(figsize=(20, 12))
ax.imshow(side_by_side)
ax.set_title("Original (left) vs OCR extraction colored by confidence (right)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

---

## Summary: All five images — raw vs smart vs full

Compare three OCR modes across all images:
- **Raw**: no preprocessing at all
- **Smart**: auto-detected preprocessing (only what each image needs)
- **Full**: all preprocessing forced on (CLAHE + upscale + deskew regardless)

In [ ]:
rows = []

for name, meta in manifest.items():
    img = cv2.imread(str(images_dir / meta["filename"]))
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = g.shape

    rec = analyze_image(g, config)
    smart_img = preprocess(g, rec)

    # Full = force all preprocessing on
    full_recipe = {"scale": rec["scale"], "apply_clahe": True,
                   "clahe_clip": rec["clahe_clip"], "skew_angle": rec["skew_angle"]}
    full_img = preprocess(g, full_recipe)

    raw_r = run_tesseract(g)
    smart_r = run_tesseract(smart_img)
    full_r = run_tesseract(full_img)

    best_pass = max(["raw", "smart", "full"],
                    key=lambda k: {"raw": raw_r, "smart": smart_r, "full": full_r}[k]["avg_confidence"])
    best_conf = {"raw": raw_r, "smart": smart_r, "full": full_r}[best_pass]["avg_confidence"]

    if best_conf >= accept_thresh:
        route = "ACCEPT"
    elif best_conf >= review_thresh:
        route = "REVIEW"
    else:
        route = "ESCALATE"

    rows.append({
        "name": name, "size": f"{w}×{h}",
        "scale": rec["scale"], "clahe": rec["apply_clahe"],
        "deskew": abs(rec["skew_angle"]) > 0.3,
        "raw": raw_r["avg_confidence"],
        "smart": smart_r["avg_confidence"],
        "full": full_r["avg_confidence"],
        "best": best_pass, "routing": route,
    })

print(f"{'Image':25s} {'Size':>10s} {'Scl':>4s} {'CLAHE':>6s} {'Dskw':>5s} "
      f"{'Raw%':>6s} {'Smart%':>7s} {'Full%':>7s} {'Best':>6s} {'Route':>9s}")
print("─" * 100)
for r in rows:
    print(f"{r['name']:25s} {r['size']:>10s} {r['scale']:>3d}x {str(r['clahe']):>5s} "
          f"{str(r['deskew']):>5s} {r['raw']:>5.1f}% {r['smart']:>6.1f}% "
          f"{r['full']:>6.1f}% {r['best']:>6s} {r['routing']:>9s}")

### Confidence comparison: raw vs smart vs full

In [ ]:
names = [r["name"] for r in rows]
raw_c = [r["raw"] for r in rows]
smart_c = [r["smart"] for r in rows]
full_c = [r["full"] for r in rows]

x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width, raw_c, width, label="Raw", color="steelblue", alpha=0.7)
ax.bar(x, smart_c, width, label="Smart", color="coral", alpha=0.7)
ax.bar(x + width, full_c, width, label="Full", color="mediumpurple", alpha=0.7)

ax.axhline(accept_thresh, color="green", ls="--", alpha=0.5, label=f"Accept ({accept_thresh}%)")
ax.axhline(review_thresh, color="orange", ls="--", alpha=0.5, label=f"Review ({review_thresh}%)")

ax.set_ylabel("Avg word confidence (%)")
ax.set_title("OCR confidence by preprocessing mode")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha="right")
ax.legend(loc="lower right")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()